In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np

from matplotlib import pyplot as plt
from matplotlib_venn import venn3

import statsmodels.api as sm

import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import descri_function as des_fun

from pathlib import Path

from datetime import datetime

In [2]:
dta = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/cities_valid_for_MEM_26_03_2026.parquet')

# Read data with EARS output

In [3]:
df = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/aesop_with_MEM_26_03_2026_earspar.parquet')

In [4]:
df.columns

Index(['co_ibge', 'epiweek', 'year', 'year_week', 'week', 'atend_ivas',
       'atend_totais', 'mem_surge_01_correct_with_consec',
       'warning_final_mem_surge_01', '10w001', '10w005', '10w010', '10w025',
       '10w050', '10w100', '10w150', '6w001', '6w005', '6w010', '6w025',
       '6w050', '6w100', '6w150', '8w001', '8w005', '8w010', '8w025', '8w050',
       '8w100', '8w150'],
      dtype='object')

In [5]:
# Select cities for the manuscript analysis (valid MEM)
lst = list(set(df.co_ibge.unique()) - set(dta.co_ibge.unique()))
df = df[~df.co_ibge.isin(lst)]

df =  df[(df.year_week >= '2022-42') &(df.year_week <= '2025-32')]

In [6]:
df.co_ibge.nunique()

5365

In [7]:
df_mem = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/mem_output_26_03_2026.parquet')

df_mem = df_mem[df_mem.epiyear.isin([2022,2023, 2024,2025])]

dta_intens = df_mem.groupby(['co_ibge'])[['baseline', 'post_baseline', 'epidemic_threshold',
       'post_threshold', 'low_level', 'medium_level', 'high_level']].max().reset_index()

# FUNCTION TO ASSIGN INTENSITY

def classify_block(values):
    
    # Check from most severe to least
    if (values > high).any():
        return 'very high'
    
    elif (values > med).any():
        return 'high'
    
    elif (values > low).any():
        return 'medium'
    
    elif (values > epi).any():
        return 'low'
    
    elif ((values > base) & (values <= epi)).any():
        return 'very low'
    
    elif (values <= base).any():
        return 'baseline'
    
    return np.nan


import warnings
warnings.filterwarnings('ignore')

lst = []

for code in df.co_ibge.unique():

    #print(code)

    set_muni = df[df.co_ibge == code].copy()
    set_muni_inten = dta_intens[dta_intens.co_ibge == code]

    thresholds = set_muni_inten.iloc[0]

    base = int(thresholds['baseline'])
    epi = int(thresholds['epidemic_threshold'])
    low  = int(thresholds['low_level'])    if pd.notna(thresholds.get('low_level'))    else 2 * epi
    med  = int(thresholds['medium_level']) if pd.notna(thresholds.get('medium_level')) else 4 * epi
    high = int(thresholds['high_level'])   if pd.notna(thresholds.get('high_level'))   else 6 * epi
    very_high = thresholds.get('very_high_level', np.inf)


    set_muni = set_muni.sort_values(['year_week'])

    # IDENTIFY BLOCKS

    flag = set_muni['mem_surge_01_correct_with_consec']

    # Create block IDs for consecutive 1s
    set_muni['block'] = (flag != flag.shift()).cumsum()

    # Keep only blocks where flag == 1
    set_muni['intensity'] = np.nan


    #  APPLY TO EACH BLOCK

    for block_id, group in set_muni.groupby('block'):
    
        if group['mem_surge_01_correct_with_consec'].iloc[0] != 1:
            continue
    
        values = group['atend_ivas']
    
        intensity_label = classify_block(values)
    
        set_muni.loc[group.index, 'intensity'] = intensity_label

    lst.append(set_muni)

result = pd.concat(lst)
result['mem_low'] = ((result['intensity'] == 'low') & (result['warning_final_mem_surge_01'] == 1)).astype(int)
result['mem_medium'] = ((result['intensity'] == 'medium') & (result['warning_final_mem_surge_01'] == 1)).astype(int)
result['mem_high'] = ((result['intensity'] == 'high') & (result['warning_final_mem_surge_01'] == 1)).astype(int)
result['mem_very_high'] = ((result['intensity'] == 'very high') & (result['warning_final_mem_surge_01'] == 1)).astype(int)

In [8]:
# select period

data = result#[result.year_week >= '2025-01']


models = [
    '10w001', '10w005', '10w010', '10w025',
       '10w050', '10w100', '10w150', '6w001', '6w005', '6w010', '6w025',
       '6w050', '6w100', '6w150', '8w001', '8w005', '8w010', '8w025', '8w050',
       '8w100', '8w150'
]

warn_col = 'warning_final_mem_surge_01'

warn_col_with_consec = 'mem_surge_01_correct_with_consec'

In [9]:
lst = []

for sel_model in models:

    df_warning_count = des_fun.antici_count(data, sel_model, warn_col,warn_col_with_consec, 'co_ibge')
    performance_summary = des_fun.summarize_performance(df_warning_count)

    performance_summary['model'] = sel_model

    lst.append(performance_summary)
        
summary =  pd.concat(lst)

In [23]:
df_metrics = summary.copy()

metrics = ['Sensitivity ', 'Specificity', 'PPV', 'NPV','n3','n2','n1','n0']

df_metrics = df_metrics[df_metrics['Metric'].isin(metrics)]

In [24]:
df_metrics.head()

,Metric,Value,model
4,Sensitivity,35.3%,10w001
5,Specificity,97.1%,10w001
6,PPV,90.9%,10w001
7,NPV,97.0%,10w001
27,n3,1648,10w001


In [25]:
# Convert % to numeric
df_metrics['value_num'] = (
    df_metrics['Value']
    .str.replace('%', '')
    .astype(float)
) 

df_metrics = df_metrics.pivot_table(
    index=['model'],
    columns='Metric',
    values='value_num'
).reset_index()

In [26]:
df_metrics = df_metrics.assign(NPV = df_metrics['NPV'] / 100, 
                               PPV = df_metrics['PPV'] / 100,
                               Sensitivity = df_metrics['Sensitivity '] / 100, 
                               Specificity = df_metrics['Specificity'] / 100)

In [27]:
df_metrics = df_metrics[['model', 'NPV', 'PPV', 'Specificity', 'n0', 'n1', 'n2',
       'n3', 'Sensitivity']]

In [28]:
df_metrics = df_metrics.assign(J = df_metrics['Sensitivity'] + df_metrics['Specificity'] -1,
                              w = 0.75*df_metrics.n3 + 0.20*df_metrics.n2 + 0.04*df_metrics.n1 + 0.01*df_metrics.n0)

In [29]:
#df_metrics.sort_values(by="w", ascending=False).to_csv('best_ears.csv')

In [30]:
df_metrics.sort_values(by="J", ascending=False)

Metric,model,NPV,PPV,Specificity,n0,n1,n2,n3,Sensitivity,J,w
6,10w150,0.989,0.704,0.853,8672.0,2348.0,2474.0,7502.0,0.795,0.648,6301.94
20,8w150,0.989,0.687,0.841,8668.0,2439.0,2509.0,7538.0,0.801,0.642,6339.54
13,6w150,0.990,0.666,0.824,8637.0,2492.0,2626.0,7672.0,0.811,0.635,6465.25
5,10w100,0.987,0.740,0.880,8893.0,2213.0,2140.0,6289.0,0.740,0.620,5322.20
19,8w100,0.987,0.723,0.869,8920.0,2274.0,2205.0,6371.0,0.749,0.618,5399.41
12,6w100,0.987,0.700,0.853,8938.0,2331.0,2336.0,6567.0,0.764,0.617,5575.07
11,6w050,0.984,0.745,0.885,8966.0,2083.0,1933.0,5269.0,0.691,0.576,4511.33
18,8w050,0.983,0.769,0.900,8905.0,2003.0,1769.0,4947.0,0.667,0.567,4233.22
4,10w050,0.983,0.788,0.912,8880.0,1905.0,1704.0,4812.0,0.655,0.567,4114.80
10,6w025,0.982,0.777,0.907,8763.0,1869.0,1645.0,4329.0,0.629,0.536,3738.14


In [31]:
#summary[summary.model == '6w150']

In [32]:
#summary.to_csv('summary_ears.csv')

# Salvar o dado com o melhor modelo 

In [33]:
result.columns

Index(['co_ibge', 'epiweek', 'year', 'year_week', 'week', 'atend_ivas',
       'atend_totais', 'mem_surge_01_correct_with_consec',
       'warning_final_mem_surge_01', 'sinal_ears_atend'],
      dtype='object')

In [34]:
result = result[['co_ibge', 'epiweek', 'year', 'year_week', 'week', 'atend_ivas',
       'atend_totais', 'mem_surge_01_correct_with_consec',
       'warning_final_mem_surge_01', '10w150']]

In [35]:
result = result.rename(columns = {'10w150': 'sinal_ears_atend'})

In [36]:
result.columns

Index(['co_ibge', 'epiweek', 'year', 'year_week', 'week', 'atend_ivas',
       'atend_totais', 'mem_surge_01_correct_with_consec',
       'warning_final_mem_surge_01', 'sinal_ears_atend'],
      dtype='object')

In [37]:
out_dir = Path("/opt/storage/shared/aesop/aesop_shared/ensamble_modelling")


fname = f"aesop_{datetime.now():%d_%m_%Y}_with_MEM_ears.parquet"

result.to_parquet(out_dir / fname)